# IPARD III — Embedding Oluşturma

**Model:** `ytu-ce-cosmos/turkish-e5-large`

**Önemli:** Bu model E5 mimarisini kullanır.
- Chunk'lar (passage) encode edilirken `passage:` prefix'i eklenir.
- Sorgular encode edilirken `query:` prefix'i eklenir (`rag_pipeline.py`'de).
- İki prefix farklı olduğu için embedding'leri **yeniden üretmek** gerekir.

**Adımlar:**
1. `all_chunks.json` dosyasını Colab'a yükle
2. Hücreleri sırayla çalıştır
3. Üretilen `embeddings.npy` dosyasını indir, `data/` klasörüne koy

In [ ]:
# Gerekli kütüphaneyi kur
!pip install sentence-transformers -q

In [ ]:
import json
import time
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "ytu-ce-cosmos/turkish-e5-large"

print(f"Model yükleniyor: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print("Model hazır.")

# all_chunks.json dosyasını Colab'a yüklemeyi unutma!
with open("/content/all_chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Toplam chunk: {len(chunks)}")

In [ ]:
# ── Text hazırlama ────────────────────────────────────────────
# turkish-e5-large için passage prefix'i ZORUNLU.
# Sorgularda rag_pipeline.py içinde "query: " prefix'i ekleniyor.
# Burada chunk'lar için "passage: " prefix'i ekliyoruz.
# Bu ikili asimetri olmadan cosine similarity düşük kalır!

texts = []
for c in chunks:
    enriched = c.get("enriched_text", "")
    if enriched:
        body = enriched[:600]   # ~600 char ≈ ~150 token, E5 için ideal
    else:
        heading = c.get("heading", "")
        text    = c.get("text", "")
        body    = f"{heading}\n{text}" if heading else text
        body    = body[:600]

    # E5 passage prefix — KRİTİK
    texts.append(f"passage: {body}")

print(f"Encode edilecek metin sayısı: {len(texts)}")
print(f"\nÖrnek (ilk metin):\n{texts[0][:300]}")

In [ ]:
# ── Embedding üretimi ─────────────────────────────────────────
print("Embedding başlıyor...")
t0 = time.time()

embeddings = model.encode(
    texts,
    batch_size=64,           # Colab T4 GPU için 64, CPU ise 16 kullan
    show_progress_bar=True,
    normalize_embeddings=True,  # Cosine similarity için normalize et
)

elapsed = time.time() - t0
print(f"Tamamlandı: {elapsed:.1f} saniye")
print(f"Embedding shape: {embeddings.shape}")
print(f"Chunk sayısı eşleşiyor mu: {len(chunks) == len(embeddings)}")

In [ ]:
# ── Kaydet ───────────────────────────────────────────────────
np.save("/content/embeddings.npy", embeddings)
print("Kaydedildi: /content/embeddings.npy")
print("Bu dosyayı indirip projenin data/ klasörüne koy.")

# Colab'dan otomatik indirme
from google.colab import files
files.download("/content/embeddings.npy")

## Sonraki adımlar

1. `embeddings.npy` dosyasını `data/` klasörüne koy
2. `python build_db.py` komutunu çalıştır (ChromaDB'yi sıfırdan yeniden oluşturur)
3. `uvicorn api:app --reload --port 8000` ile API'yi başlat
4. `streamlit run app.py` ile arayüzü aç

**Not:** `build_db.py` çalıştırılmadan önce eski `data/chromadb/` klasörünü silmek iyi bir pratiktir.